# Plant-ID: distil BioCLIP into a phone-sized student (Colab)

Trains the MobileCLIP2-S0 student and exports it to ONNX without touching the
garden app's database. Everything comes from a bundle built on the laptop by
`ml/data/build_plantid_colab_bundle.py`, uploaded to the app's **private** bucket at
`gs://garden-app-static/plantid-colab/`. Background: `ml/docs/PLANT_ID_MODEL.md`.

**Before running:** *Runtime → Change runtime type → T4 GPU.* Sign in with a Google
account that has access to the `innate-conquest-491313-k3` project.

Checkpoints sync to `gs://garden-app-static/plantid-colab/out/` every 2 minutes. If the
session drops, reconnect and re-run all cells: training resumes from the last epoch.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'

In [ ]:
from google.colab import auth
auth.authenticate_user()
GCS = 'gs://garden-app-static/plantid-colab'
!gcloud config set project innate-conquest-491313-k3 --quiet
# Unzip to local disk: 55k small files are far faster to read locally.
!mkdir -p /content/bundle /content/out && gcloud storage cp "$GCS/plantid_colab_bundle.zip" /content/ && unzip -q -o /content/plantid_colab_bundle.zip -d /content/bundle
# Resume point from an earlier session, if there is one.
!gcloud storage rsync "$GCS/out" /content/out --recursive --quiet 2>/dev/null || echo "no earlier checkpoints"
!cat /content/bundle/meta.json

In [ ]:
!rm -rf /content/gardening_app && git clone -q --depth 1 -b plantid-vision https://github.com/gccarno/gardening_app.git /content/gardening_app
!pip install -q open_clip_torch timm onnx onnxruntime onnxscript
%cd /content/gardening_app

In [ ]:
# Teacher prototypes: BioCLIP text tower on the GPU (the laptop ships image prototypes only).
!python ml/data/build_plantid_colab_bundle.py --finish /content/bundle

## Smoke test (~2 min)
256 photos, 1 epoch, written to a scratch folder. It catches setup problems before the long run.

In [ ]:
!python ml/training/train_plantid_student.py --bundle /content/bundle --out-dir /content/smoke     --student mobileclip2_s0 --epochs 1 --max-images 256 --workers 2 --tag smoke

## Stage 1: 15 epochs (~2.5 h on a T4)
A background loop copies `/content/out` to GCS every 2 minutes. `--resume` is a no-op the
first time, then continues from the last epoch after a disconnect (and skips straight
through once stage 1 is finished). It reached calib top-1 29.9%; the teacher (BioCLIP)
scores 43.7%.

In [ ]:
sync = subprocess.Popen(['bash', '-c', f'while true; do gcloud storage rsync /content/out {GCS}/out --recursive --quiet; sleep 120; done'])
!python ml/training/train_plantid_student.py --bundle /content/bundle --out-dir /content/out     --student mobileclip2_s0 --epochs 15 --workers 2 --tag v1 --resume

## Stage 2: 15 more epochs, warm restart (~2.5 h)
Stage 1 was still improving when its cosine schedule reached zero, so this starts from its
best weights with a fresh warmup + cosine at half the learning rate. A new best is only
saved if it beats stage 1 (epoch 0 prints the starting point).

In [ ]:
!python ml/training/train_plantid_student.py --bundle /content/bundle --out-dir /content/out \n    --student mobileclip2_s0 --epochs 15 --workers 2 --tag v2 --lr 1.5e-4 --resume \n    --init-from /content/out/student_mobileclip2_s0_v1.pt

## Export: fp16 ONNX + calibration + prototypes
One model version per finished stage. Weights are stored fp16 (int8 wrecked this backbone;
see `export_plantid_onnx.py`). Fails without writing the model if it agrees with fp32 torch
on fewer than 97% of eval top-1 predictions.

In [ ]:
import os
for v in ('v1', 'v2'):
    ckpt = f'/content/out/student_mobileclip2_s0_{v}.pt'
    if os.path.exists(ckpt):
        !python ml/training/export_plantid_onnx.py --student-ckpt {ckpt}             --bundle /content/bundle --out-dir /content/out/export --version {v}
!gcloud storage rsync /content/out "$GCS/out" --recursive --quiet && gcloud storage ls -l "$GCS/out/export/"
sync.terminate()

Check each export above ends in `wrote …plantid_<v>.onnx`; the artifacts are then in
`gs://garden-app-static/plantid-colab/out/export/` for Claude Code to pull.